# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset ([source](https://doi.org/10.71728/senscience.qs2f-h81p)) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and Python tools for further analysis and machine learning tasks.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# If not already installed, install mlcroissant
!pip install -U mlcroissant

## 1. Data Loading

We'll load the dataset schema/metadata and investigate the available record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display high-level metadata
print("\033[1mDataset Name:\033[0m", dataset.metadata.name)
print("\033[1mDescription:\033[0m", dataset.metadata.description)
print("\033[1mVersion:\033[0m", dataset.metadata.version)
print("\033[1mCitation:\033[0m", dataset.metadata.cite_as)

## 2. Data Overview

Let's review the available record sets, their `@id`s, fields and columns as defined in the Croissant schema. Referencing entities by their `@id` ensures clarity and schema consistency.

In [ ]:
# List all record sets and their field @ids
print("All record sets available:")
for record_set in dataset.record_sets:
    print(f"- Record set: {record_set.id}")
    for field in record_set.fields:
        print(f"    - Field: {field.id} (Name: {field.name}, Type: {getattr(field, 'data_type', 'N/A')})")
    print('')

We will now show a sample record for each record set using the correct `@id` reference.

In [ ]:
# Print a sample record from each record set, by @id
for record_set in dataset.record_sets:
    print(f"RecordSet @id: {record_set.id}")
    # Get just one example record
    records = dataset.records(record_set=record_set.id)
    try:
        first_record = next(records)
        pprint(first_record)
    except StopIteration:
        print("  (No records found for this set)")
    print("---\n")

## 3. Data Extraction

Now, we extract the data from each record set into Pandas DataFrames, using the `@id` for each record set. For further analysis, select the main tabular data record set (by inspecting the previous overview cells).

**Note:** If your dataset contains more than one record set, use their corresponding `@id` strings seen above.

In [ ]:
# Gather all record_set @ids
record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set '{record_set_id}'.")

# Display columns and head of the main tabular data set (choose first non-empty one)
main_df_id = None
for k, df in dataframes.items():
    if not df.empty:
        main_df_id = k
        break
print(f"\nColumns in primary record set ({main_df_id}):\n", dataframes[main_df_id].columns.tolist())
dataframes[main_df_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll process the main dataframe, demonstrate filtering on a numeric field, normalization, and grouping by a key attribute.

**Note**: To follow the requirements, fields/columns should be referenced by their `@id` as defined in the metadata overview section. We'll demonstrate filtering and normalization on a numeric column (e.g., age at diagnosis or diagnosis interval, depending on what fields are available).

In [ ]:
# Let's inspect possible numeric fields/columns in the main table
main_df = dataframes[main_df_id]
print("Enumerating columns with sample values:")
sample_row = main_df.head(3)
display(sample_row)

# Try to find a numeric column -- fallback to first found
numeric_field_id = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break

# If no numeric columns found, try to cast columns named like 'age' or 'interval'
if numeric_field_id is None:
    candidates = [col for col in main_df.columns if 'age' in col.lower() or 'interval' in col.lower()]
    for c in candidates:
        try:
            main_df[c] = pd.to_numeric(main_df[c], errors='coerce')
            if main_df[c].notnull().any():
                numeric_field_id = c
                break
        except Exception:
            continue

print(f"\nUsing numeric field: {numeric_field_id}")

# Filter on this numeric value (e.g., threshold 60 for age)
threshold = main_df[numeric_field_id].quantile(0.75) if numeric_field_id else 0

filtered_df = main_df[main_df[numeric_field_id] > threshold] if numeric_field_id else pd.DataFrame()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize this numeric field
if not filtered_df.empty and numeric_field_id:
    field_normed = f"{numeric_field_id}_normalized"
    filtered_df[field_normed] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, field_normed]].head())

# Try grouping by a plausible categorical column (e.g., 'Sex', 'msi_status', etc), if present
group_field = None
candidate_groups = [col for col in main_df.columns if col.lower() in ["sex", "msi_status", "anatomical_location", "site", "metastasis"]]
for g in candidate_groups:
    if g in filtered_df.columns:
        group_field = g
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field}:")
    display(grouped_df)

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relation to a categorical group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

# If grouping field exists, plot boxplot
if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.show()

## 6. Conclusion

In this notebook, we've loaded and explored the Second Primary Colorectal Cancer survivors dataset using the `mlcroissant` library:

* Loaded metadata and found available record sets using the Croissant schema.
* Extracted main tabular data, extracted fields and performed basic data wrangling with columns referenced by their `@id`.
* Filtered and normalized a numeric field, grouped by a key attribute, and visualized distributions.

This notebook forms a starting point for deeper biomedical or ML analysis using Croissant-compliant data packages.